# Vietnamese GEC: Qwen3-4B QLoRA experiments

Clone a selected branch of **[rquq/GECC_VLSP_26](https://github.com/rquq/GECC_VLSP_26)** and execute from its repository root. Enable **Internet**, select a CUDA GPU (Kaggle **GPU T4 x2** is supported), and attach a **private** dataset containing `VGEC 2026_v1.json`. The selected branch must be pushed before running this notebook. Do not publish the dataset or credentials.

The default `MODE = "full"` runs seeds **42, 43, 44 sequentially and independently on one GPU**, with a separate split, fresh Qwen3-4B model, unadapted baseline, and three-epoch QLoRA run per seed. There is no torchrun/DDP: a second T4 can host a separate notebook experiment with a different output directory. An 8GB local CUDA GPU uses the same single-GPU launcher. Set `MODE = "smoke"` for one seed with the smoke config's training/evaluation caps. Each parameter-cell execution selects a new output folder. The public model needs no token; optional private/gated access uses Kaggle Secrets `HF_TOKEN`.



In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import subprocess
import sys

# The baseline uses PyTorch; avoid importing optional TensorFlow/Flax stacks.
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")

REPO_URL = "https://github.com/rquq/GECC_VLSP_26.git"
BRANCH = "codex/kaggle-baseline"  # Change to any published branch you want to train.
PROJECT_SUBDIR = "."  # The baseline lives at the root of this repository.
MODE = "full"  # "smoke" (one seed) or "full" (three independent seeds)
NUM_GPUS = 1  # Required: each seed runs on one GPU, never DDP.
DATA_PATH = None  # Optional exact /kaggle/input/.../VGEC 2026_v1.json path.
USE_HF_SECRET = False
CLONE_DIR = Path("/kaggle/working/gecc-source")
RUN_NAME = f"vgec-{MODE}-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')}"
OUTPUT_DIR = Path("/kaggle/working") / RUN_NAME
if MODE not in {"smoke", "full"}:
    raise ValueError("MODE must be smoke or full")
print("Branch:", BRANCH, "| output:", OUTPUT_DIR)



In [ ]:
# This cell can be rerun after a kernel restart. Git runs as a subprocess.
# Reuse only a clean checkout of the same repository and branch.
from urllib.parse import urlsplit
if urlsplit(REPO_URL).username is not None:
    raise ValueError("Do not embed GitHub credentials in REPO_URL.")
subprocess.run(["git", "check-ref-format", "--branch", BRANCH], check=True, capture_output=True)
if CLONE_DIR.exists():
    if not (CLONE_DIR / ".git").is_dir():
        raise RuntimeError(f"{CLONE_DIR} exists but is not a Git clone; choose another CLONE_DIR.")
    existing_url = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=CLONE_DIR, text=True).strip()
    if existing_url != REPO_URL:
        raise RuntimeError("CLONE_DIR belongs to a different repository; choose another CLONE_DIR.")
    existing_branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=CLONE_DIR, text=True).strip()
    if existing_branch != BRANCH:
        raise RuntimeError("CLONE_DIR uses a different branch; choose another CLONE_DIR.")
    changes = subprocess.check_output(["git", "status", "--porcelain"], cwd=CLONE_DIR, text=True)
    if changes.strip():
        raise RuntimeError("CLONE_DIR has local changes; preserve them and choose another CLONE_DIR.")

remote = subprocess.run(["git", "ls-remote", "--exit-code", "--heads", "--", REPO_URL,
                         f"refs/heads/{BRANCH}"], capture_output=True, text=True)
if remote.returncode == 2:
    raise RuntimeError(f"Branch {BRANCH!r} does not exist on GitHub. Push it or select a published BRANCH.")
if remote.returncode != 0:
    raise RuntimeError("Cannot read the Git repository. Enable Kaggle Internet and check repository access.")
if CLONE_DIR.exists():
    subprocess.run(["git", "fetch", "origin", f"refs/heads/{BRANCH}"], cwd=CLONE_DIR, check=True)
    subprocess.run(["git", "merge", "--ff-only", "FETCH_HEAD"], cwd=CLONE_DIR, check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
                    "--", REPO_URL, str(CLONE_DIR)], check=True)
PROJECT_DIR = (CLONE_DIR / PROJECT_SUBDIR).resolve()
if not PROJECT_DIR.is_relative_to(CLONE_DIR.resolve()):
    raise ValueError("PROJECT_SUBDIR must stay inside the cloned repository.")
for filename in ["requirements.txt", "vgec/train.py", "vgec/experiment.py", "tools/kaggle_run.py", "configs/smoke.json", "configs/baseline.json"]:
    if not (PROJECT_DIR / filename).is_file():
        raise FileNotFoundError(f"Missing {filename} in {PROJECT_DIR}; check the selected branch and PROJECT_SUBDIR.")
SOURCE_INFO = {"repo_url": REPO_URL, "branch": BRANCH, "project_subdir": PROJECT_SUBDIR,
               "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CLONE_DIR, text=True).strip()}
print(json.dumps(SOURCE_INFO, indent=2))



In [ ]:
# Keep the CUDA-enabled torch provided by Kaggle. A constraint prevents
# pip from replacing it while installing the pinned Hugging Face stack.
from importlib.metadata import version
from packaging.version import Version
import tempfile

installed_torch = version("torch")
if not Version("2.6") <= Version(installed_torch) < Version("2.9"):
    raise RuntimeError(f"This baseline supports torch >=2.6,<2.9; found {installed_torch}. Use a compatible Kaggle image.")
packages = ["transformers", "peft", "accelerate", "bitsandbytes", "safetensors", "huggingface-hub", "tokenizers"]
imported_before = {name: version(name) for name in packages if name.replace("-", "_") in sys.modules}
with tempfile.TemporaryDirectory() as temporary:
    constraints = Path(temporary) / "constraints.txt"
    constraints.write_text(f"torch=={installed_torch}\n", encoding="utf-8")
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt"),
                    "--constraint", str(constraints)], cwd=PROJECT_DIR, check=True)
changed_imports = [name for name, old in imported_before.items() if version(name) != old]
if changed_imports:
    raise RuntimeError(f"Restart the kernel to reload {changed_imports}, then rerun from the parameter cell; the clean clone is reused.")
# Check the training stack directly; unrelated packages in Kaggle's image
# can have pre-existing conflicts that do not affect this PyTorch baseline.
subprocess.run([sys.executable, "-c", "import torch, transformers, peft, accelerate, bitsandbytes; print('Training imports passed')"], check=True)
print("Pinned dependencies installed; retained torch", installed_torch)



In [ ]:
if USE_HF_SECRET:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

# The launcher validates data, dependencies, CUDA devices, and the Git commit
# before downloading weights. Each seed uses one GPU sequentially and independently.
command = [sys.executable, str(PROJECT_DIR / "tools" / "kaggle_run.py"),
           "--mode", MODE, "--num-gpus", str(NUM_GPUS), "--output-dir", str(OUTPUT_DIR),
           "--expected-commit", SOURCE_INFO["commit"]]
if DATA_PATH is not None:
    command.extend(["--data", str(DATA_PATH)])
subprocess.run(command + ["--preflight-only"], cwd=PROJECT_DIR, check=True)



In [ ]:
# Evaluate each fresh unadapted model, train response-only QLoRA, reload its best
# adapter, evaluate, record source provenance, and package all seed directories.
subprocess.run(command, cwd=PROJECT_DIR, check=True)



In [ ]:
from IPython.display import JSON, FileLink, display
for filename in ["comparison.json", "source_manifest.json"]:
    print(filename)
    display(JSON(json.loads((OUTPUT_DIR / filename).read_text(encoding="utf-8"))))
archive_path = Path(str(OUTPUT_DIR) + "-artifacts.zip")
display(FileLink(str(archive_path)))
print("Save a Kaggle notebook version to retain the ZIP and run directory under Output.")



## Saved output

The ZIP contains the complete experiment root: `comparison.json`, `source_manifest.json`, and independent `seed-42/`, `seed-43/`, `seed-44/` directories for full mode (only `seed-42/` for smoke). Every seed saves its best adapter/tokenizer, unadapted baseline predictions/metrics, adapted predictions/metrics, identity metrics, epoch history, and run manifest. Metrics include clean-input preservation, overcorrection, invalid JSON, and reference-slice diagnostics. Source identity is attached to the aggregate comparison and each seed manifest; model and dataset hashes are recorded per seed. The ZIP is private run output, not an official submission.

For another run, rerun all cells to generate a new output directory. The source cell fetches updates to the same branch and refuses local edits or a different repository/branch. For a branch switch or a force-pushed branch, choose a new `CLONE_DIR`. Full runs use `configs/baseline.json`. Predictions use a local proposed format; the competition's official submission format is still unconfirmed.
